In [1]:
!pip install wandb


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/icml_face_data.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/fer2013.tar.gz
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/example_submission.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/test.csv


In [3]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.layers import RandomRotation, RandomFlip, RandomZoom

def create_deeper_cnn():
    """
    Deeper CNN with built-in augmentation layers
    """
    model = Sequential([
        # Augmentation Layers (Active only during training)
        RandomRotation(0.1, input_shape=(48, 48, 1)), # ~10 degrees
        RandomFlip('horizontal'),
        RandomZoom(0.1),
        
        # Block 1
        Conv2D(32, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        
        # Block 2
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        
        # Flatten and Dense Layers
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(7, activation='softmax')
    ])
    
    return model


2026-06-16 17:31:24.463443: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781631084.646732      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781631084.700622      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781631085.138602      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781631085.138638      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781631085.138641      58 computation_placer.cc:177] computation placer alr

In [4]:

model = create_deeper_cnn()
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
I0000 00:00:1781631110.088279      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1781631110.094150      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_rotation                 │ (None, 48, 48, 1)      │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 48, 48, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 48, 48, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 46, 46, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 23, 23, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 21, 21, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6400)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       819,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 839,047 (3.20 MB)

 Trainable params: 839,047 (3.20 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load training data
train_df = pd.read_csv('/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv')

# Convert pixel strings to arrays and reshape to 48x48 images
pixels = train_df['pixels'].apply(lambda x: np.array(x.split(), dtype=np.float32))
images = np.stack(pixels).reshape(-1, 48, 48, 1)

# Normalize pixel values to 
images = images / 255.0

# Labels (0-6)
labels = train_df['emotion'].values

# Split into training and validation sets (80% train, 20% val)
X_train, X_val, y_train, y_val = train_test_split(images, labels, test_size=0.2, random_state=42, stratify=labels)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")


Training set shape: (22967, 48, 48, 1)
Validation set shape: (5742, 48, 48, 1)


In [7]:
import wandb
import tensorflow as tf

# 1. Login
wandb.login(key="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")

# 2. Initialize the run
run = wandb.init(
    entity="slomi23-free-university-of-tbilisi-",
    project="fer2013-cnn",
    dir="/kaggle/working",
    mode="online", 
    config={
        "learning_rate": 0.001,
        "architecture": "SimpleCNN",
        "dataset": "FER2013",
        "epochs": 50,
        "batch_size": 32,
    }
)

print(f"W&B Run URL: {run.url}")

# 3. Use WandbMetricsLogger instead of WandbCallback
# This logs metrics (loss, accuracy) without trying to log the model graph
wandb_metrics_logger = wandb.keras.WandbMetricsLogger()

# 4. Define Early Stopping
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True
)

# 5. Train the model
history = model.fit(
    X_train, y_train, 
    batch_size=32,
    epochs=50,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, wandb_metrics_logger]
)

# 6. Log the final model weights manually if needed
# wandb.save("model.h5") # You would need to save the model first

# 7. Finish the run
wandb.finish()


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


W&B Run URL: https://wandb.ai/slomi23-free-university-of-tbilisi-/fer2013-cnn/runs/7ozoab50
Epoch 1/50


I0000 00:00:1781631224.079832     133 cuda_dnn.cc:529] Loaded cuDNN version 91002


718/718 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.2593 - loss: 1.7927 - val_accuracy: 0.3159 - val_loss: 1.7128
Epoch 2/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.3209 - loss: 1.7072 - val_accuracy: 0.3497 - val_loss: 1.6021
Epoch 3/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.3589 - loss: 1.6402 - val_accuracy: 0.4253 - val_loss: 1.5064
Epoch 4/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.3793 - loss: 1.5854 - val_accuracy: 0.4431 - val_loss: 1.4518
Epoch 5/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.3929 - loss: 1.5582 - val_accuracy: 0.4448 - val_loss: 1.4303
Epoch 6/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.4061 - loss: 1.5338 - val_accuracy: 0.4481 - val_loss: 1.4246
Epoch 7/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.4124 - loss: 1.5131 - val_accuracy: 0.4692 - val_loss: 1.3899
Epoch 8/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.4193 - loss: 1.4907 - val_accuracy: 0.4659 - val

epoch/accuracy,▁▃▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇█▇████████
epoch/epoch,▁▁▂▂▂▂▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇██
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▇▆▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
epoch/val_accuracy,▁▂▅▅▅▅▆▆▆▇▇▇▇▇▇▇███▇███████
epoch/val_loss,█▆▅▄▄▄▃▃▃▂▂▂▂▂▂▂▁▁▂▂▁▁▁▁▁▁▁
epoch/accuracy,0.48822
epoch/epoch,26
epoch/learning_rate,0.001
epoch/loss,1.33763
epoch/val_accuracy,0.52438


In [8]:
val_loss, val_acc = model.evaluate(X_val, y_val)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")


180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5272 - loss: 1.2499
Validation Loss: 1.2499
Validation Accuracy: 0.5272
